# EWC: Evaluación Class-IL

Este notebook implementa **Elastic Weight Consolidation** en el escenario **Class-IL**.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader
from models import CNN, ClassIncrementalClassifier
from dataloaders import SequentialCIFAR10
from utils_class_il import evaluate_class_il
from copy import deepcopy

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 128
LAMBDA_EWC = 1000
EPOCHS = 10
print(f"Usando dispositivo: {device}")

Usando dispositivo: mps


In [2]:
def compute_fisher(model, train_loader, device):
    model.eval()
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
    criterion = nn.CrossEntropyLoss()
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        
        model.zero_grad()
        loss.backward()
        
        for n, p in model.named_parameters():
            if p.grad is not None:
                fisher[n] += p.grad.pow(2) * len(x)
                
    for n in fisher:
        fisher[n] /= len(train_loader.dataset)
    return fisher

def ewc_loss(model, current_loss, fisher, params_star, lambda_ewc):
    penalty = 0
    for n, p in model.named_parameters():
        if n in fisher:
            penalty += (fisher[n] * (p - params_star[n]).pow(2)).sum()
    return current_loss + lambda_ewc * penalty

In [3]:
seq_cifar = SequentialCIFAR10(batch_size=BATCH_SIZE)
backbone = CNN(in_channels=3, embedding_dim=32)
model = ClassIncrementalClassifier(backbone, embedding_dim=32, total_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

fisher_dict = {}
params_star_dict = {}

for task_id in range(5):
    print(f"\n--- ENTRENANDO TAREA {task_id} ---")
    model.add_task(seq_cifar.task_classes[task_id])
    train_ds = seq_cifar.get_task_train_dataset(task_id, remap_labels=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = nn.CrossEntropyLoss()(logits, y)
            
            if task_id > 0:
                loss = ewc_loss(model, loss, fisher_dict, params_star_dict, LAMBDA_EWC)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f}")
    
    # Actualizar EWC
    print(f"  Calculando matriz de Fisher...")
    fisher_dict = compute_fisher(model, train_loader, device)
    params_star_dict = {n: p.clone().detach() for n, p in model.named_parameters()}
    
    # Evaluar Class-IL
    all_test = ConcatDataset([seq_cifar.get_task_test_dataset(tid) for tid in range(task_id+1)])
    acc = evaluate_class_il(model, DataLoader(all_test, batch_size=BATCH_SIZE), device, [])
    print(f"Precisión Class-IL tras Tarea {task_id}: {acc:.2f}%")


--- ENTRENANDO TAREA 0 ---
  Epoch 1/10 | Loss: 0.5187
  Epoch 2/10 | Loss: 0.4098
  Epoch 3/10 | Loss: 0.3667
  Epoch 4/10 | Loss: 0.3227
  Epoch 5/10 | Loss: 0.2997
  Epoch 6/10 | Loss: 0.2799
  Epoch 7/10 | Loss: 0.2638
  Epoch 8/10 | Loss: 0.2563
  Epoch 9/10 | Loss: 0.2471
  Epoch 10/10 | Loss: 0.2420
  Calculando matriz de Fisher...
Precisión Class-IL tras Tarea 0: 91.85%

--- ENTRENANDO TAREA 1 ---
  Epoch 1/10 | Loss: 0.7097
  Epoch 2/10 | Loss: 0.5849
  Epoch 3/10 | Loss: 0.5591
  Epoch 4/10 | Loss: 0.5693
  Epoch 5/10 | Loss: 0.5589
  Epoch 6/10 | Loss: 0.5466
  Epoch 7/10 | Loss: 0.5542
  Epoch 8/10 | Loss: 0.5554
  Epoch 9/10 | Loss: 0.5339
  Epoch 10/10 | Loss: 0.5443
  Calculando matriz de Fisher...
Precisión Class-IL tras Tarea 1: 37.52%

--- ENTRENANDO TAREA 2 ---
  Epoch 1/10 | Loss: 0.8658
  Epoch 2/10 | Loss: 0.5547
  Epoch 3/10 | Loss: 0.5213
  Epoch 4/10 | Loss: 0.5047
  Epoch 5/10 | Loss: 0.4966
  Epoch 6/10 | Loss: 0.4973
  Epoch 7/10 | Loss: 0.4800
  Epoch 8/10